# 📝 추출 품질 과제 LV2: 오류를 교정하고 전후를 비교하기

교안 02의 절 순서를 그대로 따릅니다.  
**오류 원문 확인 → 모순 후보 검사 → 한 항목 교정 → 같은 골드로 재평가 → 좋아진 점과 나빠진 점 판단** 순서입니다.  
기존 자가면역 논문 데이터에서 **주석된 두 문서만** 평가하고, 잘못 뽑힌 것과 빠뜨린 것의 근거를 읽습니다.

- 코딩 9문항과 서술형 3문항입니다. 위에서부터 작성합니다.  
- 입력은 `lv2_corpus.jsonl`, `lv2_triples.jsonl`, `lv2_gold.jsonl`, `lv2_gold_checklist.jsonl`입니다.  
- 교정은 **저장된 자료를 사람이 고치는 작업**입니다. 새 프롬프트나 모델을 실행한 결과가 아니므로 모델 호출은 없습니다.  
- 변수에 담는 **모든 비율은 반올림하지 않은 나눗셈 결과**입니다. 자릿수를 줄이는 것은 화면 출력에서만 합니다.  
- 다른 문서의 골드나 교안의 추출 결과로 대체하지 않습니다.

In [ ]:
# [제공 코드]

# 실습에 공통으로 쓸 파일 경로와 읽기, 저장 함수를 준비합니다.

import json
import random
from collections import Counter
from pathlib import Path

data_dir = Path("data")  # 제공된 원문, 추출된 트리플, 골드 파일이 있는 폴더입니다.
output_dir = Path("output")  # 직접 계산한 지표와 검토 기록을 저장할 폴더입니다.
output_dir.mkdir(exist_ok=True)

def load_rows(filename):
    """data 폴더의 JSONL 파일을 딕셔너리 목록으로 읽습니다."""
    return [json.loads(line) for line in (data_dir / filename).read_text(encoding="utf-8").splitlines() if line.strip()]

def write_json(filename, value):
    """이번 실습의 결과를 output 폴더에 JSON으로 저장합니다."""
    (output_dir / filename).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_rows(filename, rows):
    """검토 기록을 한 줄에 한 항목인 JSONL로 저장합니다."""
    content = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    (output_dir / filename).write_text(content + "\n", encoding="utf-8")

def triple_key(row):
    """고유 관계를 비교할 (주어, 관계, 목적어) 튜플을 돌려줍니다."""

    # 평가 전에 표기를 바꾸지 않습니다. 이름 정규화는 다음 단원에서 배웁니다.
    return (row["subject"], row["relation"], row["object"])

In [ ]:
# [제공 코드]

# 논문 추출을 관계, 타입 규칙에 따라 통과와 기각으로 나눌 함수를 준비합니다.

signatures = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

def check_signature(row):
    """관계, 타입 위반 사유를 돌려주고, 통과하면 None을 돌려줍니다."""
    if row["relation"] not in signatures:
        return "허용 관계가 아님"
    # signatures의 값은 해당 관계가 요구하는 주어 타입과 목적어 타입입니다.
    subject_type, object_type = signatures[row["relation"]]
    if row["subject_type"] != subject_type:
        return f"주어 타입이 {subject_type} 이어야 함"
    if row["object_type"] != object_type:
        return f"목적어 타입이 {object_type} 이어야 함"

    return None

def split_schema(rows):
    """추출 목록을 스키마 통과 목록과 사유가 붙은 기각 목록으로 나눕니다."""
    valid, rejected = [], []
    for row in rows:
        reason = check_signature(row)
        if reason is None:
            valid.append(row)
        else:
            # 원본은 유지하고 기각 목록에만 사유를 덧붙입니다.
            rejected.append(dict(row, reject_reason=reason))

    return valid, rejected

In [ ]:
# [제공 코드]

# 추출과 골드를 비교해 TP, FP, FN, 정밀도, 재현율, F1을 계산할 함수를 정의합니다.

def measure_exact(rows, gold_rows):
    """고유 관계의 완전일치 TP, FP, FN과 정밀도, 재현율, F1을 돌려줍니다."""
    # 집합으로 바꿔 같은 관계를 여러 번 뽑아도 한 번만 셉니다.
    predicted = {triple_key(row) for row in rows}
    expected = {triple_key(row) for row in gold_rows}
    tp = len(predicted & expected)  # 추출 결과와 골드 양쪽에 있는 관계입니다.
    fp = len(predicted - expected)  # 골드에 없는 추출입니다. 표기 차이도 포함합니다.
    fn = len(expected - predicted)  # 골드에는 있지만 추출하지 못한 관계입니다.
    # 분모가 없으면 0점 대신 미산출(None)로 남깁니다.
    precision = tp / len(predicted) if predicted else None
    recall = tp / len(expected) if expected else None

    # 골드가 있는데 아무것도 뽑지 않으면 F1은 0입니다. 골드가 없으면 평가에서 별도 표시합니다.
    f1 = 2 * tp / (2 * tp + fp + fn) if expected else None

    return {"predicted": len(predicted), "gold": len(expected), "tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}

In [ ]:
# [제공 코드] 골드가 없는 네 문서는 평가 대상에 포함하지 않습니다.
corpus = load_rows("lv2_corpus.jsonl")
docs = {row["doc_id"]: row for row in corpus}
triples = load_rows("lv2_triples.jsonl")
valid, rejected = split_schema(triples)
gold = load_rows("lv2_gold.jsonl")
checklist = load_rows("lv2_gold_checklist.jsonl")

# 양성 골드에서 문서 범위를 역으로 만들지 않고 주석 대상 목록을 별도로 둡니다.
# 그래야 나중에 관계가 0건인 주석 문서도 평가 범위에서 사라지지 않습니다.
annotated_doc_ids = {"PMC13493376", "PMC13495420"}
evaluation_relations = {"TREATS", "PALLIATES", "BINDS", "UPREGULATES_CG",
                        "DOWNREGULATES_CG", "ASSOCIATES", "PRESENTS"}
print(f"추출 {len(triples)}행 · 스키마 통과 {len(valid)}행 · 골드 {len(gold)}항목")
for doc_id in sorted(annotated_doc_ids):
    print(doc_id, docs[doc_id]["title"])

**이번 평가의 기준**

- 평가 대상은 지정한 문서와 관계 범위입니다. 범위 밖 결과는 먼저 분리합니다.  
- 한 항목은 고유한 **(주어, 관계, 목적어)**입니다. 같은 관계가 여러 문서에 있어도 한 번만 셉니다. 출처와 근거는 별도 기록으로 보존합니다.  
- 세 값이 **문자열까지 모두 같을 때** 일치로 셉니다. 대소문자·괄호가 다르면 다른 항목입니다.  
- 이 점수는 **저장된 골드와의 완전일치**입니다. FP라고 해서 반드시 원문의 의미를 틀리게 읽었다는 뜻은 아닙니다. 원문 근거와 표기 차이를 함께 확인합니다.  
- 스키마 검사, 근거 원문 일치 검사, 사람이 판정한 근거 적합 여부는 서로 다른 검사입니다. 각각의 대상과 분모를 적습니다.

**이 골드의 주석 기준을 읽으세요**

관계 7종과 문서 2편의 범위를 유지합니다. 개별 약물, 질병, 유전자, 증상만 인정하며, 약효군과 여러 유전자를 아우르는 계열 이름은 제외합니다. BINDS에는 문서가 설명하는 표적, 대사 효소, 수송체 관계도 포함합니다.

기존 골드는 문서에 적힌 이름 중 주석자가 선택한 표기를 사용했습니다. 예를 들어 `Methotrexate`와 `MTX`, `prednisone`과 `Prednisone`은 의미가 가까워도 **이번 완전일치 계산에서는 다른 문자열**입니다. 평가 도중 이름을 바꾸어 점수를 높이지 말고, 차이를 오류 근거표에 기록합니다. 표기를 통일하는 방법은 다음 단원에서 다룹니다.

골드는 절대적인 의학 지식이 아니라 이 문서, 주석 지침, 표기 선택에 따른 정답표입니다. 근거를 읽어 이견을 남길 수 있지만, 이번 비교를 계산하는 중에는 골드를 변경하지 않습니다.

## 1. 평가 범위를 맞추고 골드의 검토 범위를 확인하세요

**배경**: 전체 추출은 여섯 문서에서 나왔지만 골드는 두 문서만 주석했습니다. 교정 전후를 같은 시험으로 채점하려면 범위부터 고정해야 합니다.

**요구사항**

- **`evaluation_plan`**을 딕셔너리로 만드세요. **`doc_ids`**는 annotated_doc_ids를 정렬한 리스트, **`relations`**는 evaluation_relations를 정렬한 리스트입니다.  
- **`unit`**은 `"unique_triple"`, **`matching`**은 `"exact"`, **`gold_file`**은 `"lv2_gold.jsonl"`입니다. evaluation_plan의 키는 여기까지입니다.  
- **`scoped`**는 valid에서 해당 문서와 관계에 속하는 원본 행만 남긴 별도의 리스트입니다. 순서와 중복 근거를 유지하세요.  
- **`scope_gold`**는 gold에서 관계가 평가 범위에 있고 sources의 모든 doc_id가 주석 문서 목록에 드는 항목만 담은 별도의 리스트입니다.  
- **`gold_scope_ok`**에는 checklist의 `(doc_id, sent_id)` 집합이 주석 두 문서의 전체 문장과 같은지 확인한 불리언을 담으세요. 각 문서의 문장 번호는 **`docs[doc_id]["sentences"]`**의 개수로 만들고 sent_id는 0부터 시작합니다.

**확인 기준**: scoped는 9행이고 scope_gold는 7항목입니다. gold_scope_ok는 True이며 검토한 문장은 30개입니다. 같은 관계가 근거만 다르게 두 번 나온 행은 이 단계에서 보존합니다.

<details><summary>힌트</summary>

**접근방법**: 평가 계획을 적고 추출과 골드에 동일한 문서, 관계 범위를 적용합니다.

**세부구현**

1. 정렬한 범위를 계획에 저장하세요.  
2. 원본 추출의 source_doc_id와 relation을 함께 확인하세요.  
3. 골드 sources는 첫 항목만 보지 않고 모두 확인하세요.  
4. 문장 번호는 각 문서의 sentences 길이로 만드세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert evaluation_plan == {"doc_ids": sorted(annotated_doc_ids), "relations": sorted(evaluation_relations),
                           "unit": "unique_triple", "matching": "exact", "gold_file": "lv2_gold.jsonl"},             "다섯 키의 이름과 값을 지문 그대로 넣고 두 목록은 정렬하세요."
assert scoped == [row for row in valid if row["source_doc_id"] in annotated_doc_ids and row["relation"] in evaluation_relations],             "문서와 관계를 함께 걸러 원래 순서와 중복 근거를 유지하세요."
assert scope_gold == [row for row in gold if row["relation"] in evaluation_relations
                      and all(source["doc_id"] in annotated_doc_ids for source in row["sources"])],             "골드의 sources는 첫 항목만 보지 말고 모두 확인하세요."
assert len(scoped) == 9 and len(scope_gold) == 7, "평가 행 9개, 골드 7항목입니다."
assert gold_scope_ok is True and len(checklist) == 30, "0건 문장까지 검토 기록이 있어야 합니다."
print("✅ 평가 범위 통과!")

## 2. 완전일치 기준선을 계산하세요

**배경**: 교정 전후를 비교하려면 먼저 고칠 것이 없는 상태의 점수를 남겨야 합니다.

**요구사항**

- **`predicted_keys`**, **`gold_keys`**를 scoped와 scope_gold의 triple_key 집합으로 만드세요.  
- **`tp_keys`**, **`fp_keys`**, **`fn_keys`**에는 교집합, 추출에만 있는 차집합, 골드에만 있는 차집합을 각각 담으세요.  
- **`baseline`**에는 measure_exact로 계산한 딕셔너리를 담으세요.  
- **`duplicate_rows`**에는 scoped 행 수에서 고유 추출 관계 수를 뺀 정수를 담으세요.  
- TP, FP, FN과 P/R/F1, 중복 행 수를 출력하세요.

**확인 기준**: 고유 추출 8건, 골드 7건, TP=2, FP=6, FN=5입니다. P=0.25, R=2/7, F1=4/15이며 중복 행은 1개입니다.

<details><summary>힌트</summary>

**접근방법**: 집합으로 오류 목록을 만들고 같은 기준의 지표를 제공 함수로 계산합니다.

**세부구현**

1. 타입과 근거는 triple_key에 포함하지 않습니다.  
2. 차집합의 방향을 구분하세요.  
3. 행 수와 고유 관계 수를 따로 출력하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert predicted_keys == {triple_key(row) for row in scoped} and gold_keys == {triple_key(row) for row in scope_gold},             "두 집합은 각각 scoped와 scope_gold의 triple_key입니다."
assert tp_keys == predicted_keys & gold_keys, "TP는 두 집합의 교집합입니다."
assert fp_keys == predicted_keys - gold_keys and fn_keys == gold_keys - predicted_keys, "차집합의 방향을 확인하세요."
assert baseline == measure_exact(scoped, scope_gold), "제공 함수에 scoped와 scope_gold를 넣으세요."
assert (baseline["predicted"], baseline["gold"], baseline["tp"], baseline["fp"], baseline["fn"]) == (8, 7, 2, 6, 5),             "고유 추출 8건, 골드 7건에서 TP 2, FP 6, FN 5가 나와야 합니다."
assert abs(baseline["precision"] - 0.25) < 1e-12, "정밀도의 분모는 고유 추출 8건입니다."
assert abs(baseline["recall"] - 2 / 7) < 1e-12 and abs(baseline["f1"] - 4 / 15) < 1e-12, "재현율의 분모는 골드 7건입니다."
assert duplicate_rows == len(scoped) - len(predicted_keys) == 1, "행 수에서 고유 관계 수를 빼세요."
assert measure_exact(scoped + scoped, scope_gold) == baseline, "근거 중복으로 지표가 바뀌면 안 됩니다."
print("✅ 완전일치 기준선 통과!")

## 3. 근거가 원문과 다른 행을 찾아 교정 대상을 특정하세요

**배경**: 근거 원문 일치 검사에서 걸린 행은 관계가 틀렸다는 뜻이 아니라 인용을 확인해야 한다는 뜻입니다. 교안 02의 1-1절처럼 추출 필드와 원문을 나란히 놓고 읽습니다.

**요구사항**

- **`quote_mismatch`**를 scoped 중 근거 문자열 전체가 해당 출처 원문에 그대로 있지 않은 원본 행의 리스트로 만드세요. 대조할 원문은 문장 하나가 아니라 문서 전체인 **`docs[row["source_doc_id"]]["text"]`**입니다.  
- **`quote_case`**에 그 리스트의 첫 항목을 담으세요.  
- **`quote_source`**에는 그 근거가 요약한 원문 문장 하나를 담으세요. 해당 문서의 sentences 중 `folylpolyglutamate synthase (FPGS)`가 들어 있는 문장은 하나뿐입니다.  
- 추출된 근거와 원문 문장을 `repr`로 나란히 출력해 어디가 다른지 확인하세요.

**확인 기준**: quote_mismatch는 1행이고 그 관계는 (MTX, BINDS, FPGS)입니다. quote_source는 해당 원문에 그대로 있는 한 문장이며, 추출된 근거는 그 문장의 앞부분을 잘라내고 이어 쓴 기록입니다.

<details><summary>힌트</summary>

**접근방법**: 원문 대조가 필요한 행을 먼저 좁힌 뒤 그 근거를 뒷받침하는 원문 문장을 찾습니다.

**세부구현**

1. 근거가 비어 있지 않은지도 함께 확인하세요.  
2. 원문 문장은 in 연산으로 특정 표현이 들어 있는 것만 고르세요.  
3. repr로 출력하면 문장부호와 앞뒤 차이가 보입니다.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert quote_mismatch == [row for row in scoped
                          if not row["evidence"].strip() or row["evidence"] not in docs[row["source_doc_id"]]["text"]],             "근거가 비어 있거나 출처 원문에 그대로 없는 행만 담으세요."
assert len(quote_mismatch) == 1, "범위 안에서 인용이 어긋난 행은 한 건입니다."
assert triple_key(quote_case) == ("MTX", "BINDS", "FPGS"), "quote_case에는 quote_mismatch의 첫 항목을 담으세요."
assert quote_source == docs["PMC13493376"]["sentences"][8], "FPGS가 들어 있는 문장 하나를 고르세요."
assert quote_source in docs[quote_case["source_doc_id"]]["text"], "직접 지어 쓰지 말고 sentences에서 고른 문장을 담으세요."
assert quote_case["evidence"] not in quote_source, "추출된 근거는 원문을 그대로 옮긴 문자열이 아닙니다."
print("✅ 교정 대상 특정 통과!")

## 4. FP와 FN의 원문 근거표를 만드세요

**배경**: 점수만으로 수정 방향을 정할 수 없습니다. FP에는 모델이 적은 근거를, FN에는 골드 주석의 근거를 붙여 읽습니다.

**요구사항**

- **`fp_evidence`**를 scoped에서 fp_keys에 속하는 각 행을 변환한 딕셔너리 리스트로 만드세요. 원래 순서와 중복 근거를 보존합니다.  
- 각 항목의 **`key`**는 triple_key를 리스트로 바꾼 값, **`doc_id`**는 source_doc_id, **`evidence`**는 모델의 근거 문자열입니다.  
- 불리언 **`verbatim`**은 그 근거 문자열이 해당 원문 **`docs[doc_id]["text"]`**에 그대로 있는지 `in`으로 확인한 값입니다. 여기서는 빈 근거를 따로 가르지 않습니다. 문자열 **`status`**는 `"검토 필요"`입니다.  
- **`fn_evidence`**는 scope_gold 중 fn_keys에 드는 각 항목의 딕셔너리 리스트입니다. **`key`**는 키 리스트, **`sources`**는 골드의 sources 전체, **`status`**는 `"검토 필요"`입니다.  
- 두 목록을 출력하세요. 원문 불일치만으로 FP의 원인을 확정하지 마세요.

**확인 기준**: FP는 고유 6건이지만 근거표는 7행입니다. FN 근거표는 5행이고 각 항목의 출처를 보존합니다.

<details><summary>힌트</summary>

**접근방법**: 오류 집합으로 원본 행을 고르고 각 행의 출처 정보를 유지합니다.

**세부구현**

1. FP 행을 중복 제거하지 마세요.  
2. 출처 문서 한 편에서만 근거를 대조하세요.  
3. FN은 sources의 첫 원소가 아니라 전체를 남기세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
expected_fp = [{"key": list(triple_key(row)), "doc_id": row["source_doc_id"], "evidence": row["evidence"],
                "verbatim": row["evidence"] in docs[row["source_doc_id"]]["text"], "status": "검토 필요"}
               for row in scoped if triple_key(row) in fp_keys]
assert fp_evidence == expected_fp, "원본 순서와 모든 근거를 보존하세요."
assert fn_evidence == [{"key": list(triple_key(row)), "sources": row["sources"], "status": "검토 필요"}
                       for row in scope_gold if triple_key(row) in fn_keys],             "FN 항목은 key, sources, status 세 키만 담고 sources는 전체를 남기세요."
assert len(fp_evidence) == 7 and len(fn_evidence) == 5, "FP는 고유 6건이지만 근거 행은 7개입니다. 중복을 제거하지 마세요."
assert {tuple(row["key"]) for row in fp_evidence} == fp_keys, "key에는 triple_key를 리스트로 바꿔 담으세요."
assert sum(not row["verbatim"] for row in fp_evidence) == 1, "인용이 어긋난 FP는 한 건입니다."
print("✅ 오류 근거표 통과!")

## 5. 오류 세 건을 서로 다른 원인으로 진단하세요

**배경**: FP와 FN이라는 평가 분류와 실제 오류 원인은 다릅니다. 표기 차이와 내용 누락을 구분합니다.

**요구사항**

- **오류 진단표**를 작성하세요. 열은 `대상 / 원문 근거 / 완전일치에서의 상태 / 원인 / 다음 확인`입니다.  
- `(Prednisone, TREATS, SLE)`와 골드의 `(prednisone, TREATS, SLE)`를 한 행에서 비교하세요. 문서 PMC13495420의 문장 12를 읽습니다.  
- `(Corticosteroids, TREATS, SLE)`를 두 번째 행에 적고 같은 문장과 약효군 제외 기준을 연결하세요.  
- 골드의 `(hydroxychloroquine, TREATS, SLE)`를 세 번째 행에 적고 문장 10 및 골드 judgment를 읽으세요. 문맥에서 해석한 골드 관계라는 점과 추출 목록에 없는 점을 함께 적습니다.  
- 첫 번째 사례를 고칠 때 프롬프트 수정과 다음 단원의 이름 정규화를 구분하세요.

**확인 기준**: 표기 차이는 FP와 FN을 동시에 만들 수 있고, 약효군은 개체 범위 문제이며, 골드의 문맥 추론은 지침 검토도 필요하다고 설명합니다. 자동 채점은 하지 않습니다. 작성한 뒤 정답 노트북의 모범 서술과 비교하세요.

<details><summary>힌트</summary>

**접근방법**: 원문 구절, 판정 기준, 결론을 순서대로 연결하세요.

**세부구현**

1. 판단에 사용한 원문 표현을 짧게 인용하세요.  
2. 그 표현이 어떤 판정 기준을 충족하거나 충족하지 못하는지 설명하세요.  
3. 점수의 문제와 추출 내용의 문제를 구분하세요.

</details>

*(여기에 원문 근거와 판단을 작성하세요.)*

In [ ]:
# [제공 코드] 반대 관계 검사를 시험할 가상 기록입니다. 실제 연구 결과가 아닙니다.
# UPREGULATES_CG는 발현을 높임, DOWNREGULATES_CG는 발현을 낮춤이라는 뜻입니다.
# 두 기록은 농도와 처리 시간이 다르므로 후보를 찾았다고 해서 한쪽을 지우면 안 됩니다.
conflict_pairs = [("UPREGULATES_CG", "DOWNREGULATES_CG")]
conflict_probe = [
    {"subject": "화합물B", "subject_type": "Compound", "relation": "UPREGULATES_CG",
     "object": "유전자Y", "object_type": "Gene", "source_doc_id": "기록1",
     "evidence": "저농도로 6시간 처리한 세포에서 발현이 증가했다."},
    {"subject": "화합물B", "subject_type": "Compound", "relation": "DOWNREGULATES_CG",
     "object": "유전자Y", "object_type": "Gene", "source_doc_id": "기록2",
     "evidence": "고농도로 48시간 처리한 세포에서 발현이 감소했다."},
]

## 6. 반대 관계로 모순 후보를 찾는 검사를 만드세요

**배경**: 같은 대상에 양립하기 어려운 주장이 함께 있으면 검토 후보입니다. 조건과 시점이 다르면 두 진술이 모두 맞을 수 있으므로 자동으로 지우지 않습니다.

**요구사항**

- **`find_conflict_candidates(rows, pairs)`** 함수를 만드세요. 트리플 리스트와 서로 반대인 관계 이름 쌍의 리스트를 받습니다.  
- 같은 `(subject, object)`에 한 쌍의 두 관계가 모두 있으면 후보입니다. 후보는 딕셔너리 하나로 만드세요.  
- 각 후보의 **`pair`**는 `(subject, object)` 튜플, **`relations`**는 그 묶음에 있는 관계 이름을 중복 없이 정렬한 리스트, **`rows`**는 그 묶음에 속한 원본 행 전부를 원래 순서로 담은 리스트입니다. 반대 쌍에 없는 관계도 묶음에 있으면 함께 담습니다. 후보는 처음 나온 묶음 순서대로 담습니다.  
- **`probe_conflicts`**에 제공된 `conflict_probe`를, **`scoped_conflicts`**에 실제 `scoped`를 검사한 결과를 담으세요. 두 번 모두 제공된 **`conflict_pairs`**를 두 번째 인자로 넘깁니다.  
- **`conflict_notes`**를 probe_conflicts의 각 후보마다 `{"pair": pair를 리스트로 바꾼 값, "relations": 관계 리스트, "source_doc_ids": 각 행의 source_doc_id 리스트, "judgment": "조건 확인 필요"}` 딕셔너리로 만드세요.  
- 두 결과의 건수를 출력하세요. 후보가 0건이어도 그대로 보고합니다.

**확인 기준**: probe_conflicts는 1건이고 scoped_conflicts는 0건입니다. 이번 추출에는 발현을 높이거나 낮추는 관계가 없어서 0건이며, 이것은 모순이 없다는 증명이 아니라 이 검사로는 걸리는 것이 없다는 뜻입니다.

<details><summary>힌트</summary>

**접근방법**: 같은 주어와 목적어끼리 행을 모은 뒤 반대 관계 쌍이 모두 있는 묶음만 남깁니다.

**세부구현**

1. setdefault로 (subject, object)별 목록을 만드세요.  
2. 묶음의 관계 이름을 집합으로 만들어 쌍의 두 이름이 모두 있는지 확인하세요.  
3. 한쪽 관계만 있는 기록은 후보가 아닙니다.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert find_conflict_candidates([], conflict_pairs) == [], "빈 입력에는 후보가 없습니다."
assert find_conflict_candidates(conflict_probe[:1], conflict_pairs) == [], "한쪽 관계만 있으면 후보가 아닙니다."
assert len(probe_conflicts) == 1, "가상 기록 두 행은 한 묶음이므로 후보는 1건입니다."
assert probe_conflicts[0]["pair"] == ("화합물B", "유전자Y"), "pair는 주어와 목적어를 담은 튜플입니다."
assert probe_conflicts[0]["relations"] == ["DOWNREGULATES_CG", "UPREGULATES_CG"], "relations는 정렬한 리스트입니다."
assert probe_conflicts[0]["rows"] == conflict_probe, "rows에는 원본 행을 그대로 담으세요."
assert scoped_conflicts == [], "이번 범위에는 발현 관계가 없어 후보가 0건입니다."
assert conflict_notes == [{"pair": ["화합물B", "유전자Y"],
                           "relations": ["DOWNREGULATES_CG", "UPREGULATES_CG"],
                           "source_doc_ids": ["기록1", "기록2"], "judgment": "조건 확인 필요"}],             "네 키의 이름과 pair를 리스트로 바꾸는 것을 확인하세요."
print("✅ 모순 후보 검사 통과!")

## 7. 검토한 한 행의 한 필드만 교정하세요

**배경**: 어떤 변경 때문에 지표가 달라졌는지 알려면 한 번에 한 필드만 바꾸고 무엇을 바꿨는지 남겨야 합니다. 이번 자료는 43행이 모두 스키마를 통과해 고칠 타입 오류가 없으므로, 함수는 두 필드를 받되 실제로는 인용문만 교정합니다.

**요구사항**

- **`correct_one_field(rows, doc_id, key, field, value)`** 함수를 만드세요. key는 triple_key가 돌려주는 튜플입니다. 교정한 새 리스트와 변경 기록 딕셔너리를 튜플로 돌려줍니다.  
- field가 `"object_type"`이나 `"evidence"`가 아니면 **`ValueError`**를 발생시키세요. 이번 실습은 확인한 타입과 인용문만 교정합니다.  
- doc_id와 triple_key가 모두 같은 행이 정확히 한 개가 아니면 **`ValueError`**를 발생시키세요.  
- 원본 리스트와 원본 딕셔너리를 바꾸지 마세요. 각 행을 복사한 새 리스트에서 그 한 칸만 바꿉니다.  
- 변경 기록에는 문자열 **`source_doc_id`**, 키를 리스트로 바꾼 **`triple`**, 문자열 **`field`**, **`before`**, **`after`**와 `"원문 대조 후 사람 교정"`인 **`method`**를 담으세요.  
- **`corrected_triples`**와 **`quote_change`**에 `triples` 전체를 대상으로 quote_case의 evidence를 quote_source로 바꾼 결과를 담으세요.

**확인 기준**: corrected_triples는 43행이고 원본과 다른 곳은 한 행의 evidence 한 칸뿐입니다. triples는 그대로입니다. 같은 문서에 두 번 나오는 (Corticosteroids, TREATS, SLE)를 대상으로 부르면 ValueError가 납니다.

<details><summary>힌트</summary>

**접근방법**: 대상 행을 먼저 찾아 한 개인지 확인한 뒤 복사본에서 한 칸만 바꿉니다.

**세부구현**

1. 허용 필드가 아니면 계산을 시작하기 전에 막으세요.  
2. 문서 ID와 트리플을 함께 비교해 같은 이름이 다른 문서에 나오는 경우를 구분하세요.  
3. 리스트뿐 아니라 각 행 딕셔너리도 복사해야 원본이 함께 바뀌지 않습니다.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert triples == load_rows("lv2_triples.jsonl"), "원본 목록을 바꾸지 마세요."
assert len(corrected_triples) == len(triples) == 43, "행을 지우거나 더하지 말고 한 칸만 바꾸세요."
changed = [(i, field) for i, (left, right) in enumerate(zip(triples, corrected_triples))
           for field in left if left[field] != right[field]]
assert len(changed) == 1 and changed[0][1] == "evidence", changed
assert quote_change == {"source_doc_id": quote_case["source_doc_id"], "triple": list(triple_key(quote_case)),
                        "field": "evidence", "before": quote_case["evidence"], "after": quote_source,
                        "method": "원문 대조 후 사람 교정"},             "변경 기록의 여섯 키를 확인하고 triple은 리스트로 담으세요."
try:
    correct_one_field(triples, quote_case["source_doc_id"], triple_key(quote_case), "subject", "MTX")
except ValueError:
    pass
else:
    raise AssertionError("허용하지 않은 필드는 ValueError로 막으세요.")
try:
    correct_one_field(triples, "PMC13495420", ("Corticosteroids", "TREATS", "SLE"), "evidence", "x")
except ValueError:
    pass
else:
    raise AssertionError("대상이 두 행이면 ValueError로 막으세요.")
print("✅ 한 필드 교정 통과!")

## 8. 같은 골드로 교정 전후를 재평가하세요

**배경**: 교안 02의 4-1절처럼 원래 행에서 스키마 통과, 범위, 근거 필터를 같은 순서로 적용해 두 단계의 점수를 남깁니다.

**요구사항**

- **`evaluate_scope(rows)`** 함수를 만드세요. 추출 목록을 받아 딕셔너리 하나를 돌려줍니다.  
- 함수 안에서 split_schema로 통과 행을 고르고, annotated_doc_ids와 evaluation_relations로 범위를 맞추세요.  
- 정수 **`scoped_rows`**는 범위 안의 행 수, 정수 **`verbatim_rows`**는 그중 근거가 비어 있지 않고 원문 **`docs[...]["text"]`**에 그대로 있는 행 수, 실수 **`verbatim_rate`**는 두 값의 비율입니다. 범위 안의 행이 하나도 없으면 verbatim_rate는 **`None`**입니다.  
- **`before_quote_filter`**에는 범위 안 전체를 scope_gold와 비교한 measure_exact 결과를, **`after_quote_filter`**에는 근거 필터를 통과한 행만 비교한 결과를 담으세요.  
- **`before_scores`**에 `triples`를, **`after_scores`**에 `corrected_triples`를 넣은 결과를 담으세요.  
- 두 결과의 근거 원문 일치율과 두 단계 P/R/F1을 나란히 출력하세요.

**확인 기준**: 교정 전은 9행 중 8행이 원문 일치이고 필터 후 7관계에서 P, R, F1이 모두 2/7입니다. 교정 후는 9행 모두 원문 일치이고 필터 후 8관계에서 P=0.25, R=2/7, F1=4/15입니다. 두 결과의 필터 전 점수는 서로 같습니다.

<details><summary>힌트</summary>

**접근방법**: 검사 순서를 함수 하나에 고정해 두 목록에 똑같이 적용합니다.

**세부구현**

1. 근거 필터 전후로 measure_exact를 두 번 호출하세요.  
2. 골드는 양쪽 모두 scope_gold 하나만 씁니다.  
3. 인용을 고쳐도 주어, 관계, 목적어는 그대로라는 점을 확인하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert before_scores["scoped_rows"] == after_scores["scoped_rows"] == 9, "두 목록 모두 범위 안 9행입니다."
assert before_scores["verbatim_rows"] == 8 and before_scores["verbatim_rate"] == 8 / 9,             "교정 전에는 9행 중 8행만 원문과 같습니다. 비율은 반올림하지 마세요."
assert after_scores["verbatim_rows"] == 9 and after_scores["verbatim_rate"] == 1.0,             "교정본을 넣었는지 확인하세요."
assert before_scores["before_quote_filter"] == after_scores["before_quote_filter"] == baseline, "인용 교정은 관계 집합을 바꾸지 않습니다."
assert evaluate_scope(scoped) == before_scores, "이미 범위를 맞춘 목록을 넣어도 같은 값이어야 합니다."
empty = evaluate_scope([])
assert (empty["scoped_rows"], empty["verbatim_rows"], empty["verbatim_rate"]) == (0, 0, None),             "범위 안 행이 없으면 0으로 나누지 말고 verbatim_rate에 None을 넣으세요."
before_filter = before_scores["after_quote_filter"]
after_filter = after_scores["after_quote_filter"]
assert (before_filter["predicted"], before_filter["tp"], before_filter["fp"], before_filter["fn"]) == (7, 2, 5, 5),             "근거 필터를 통과한 행만 골드와 비교하세요."
assert abs(before_filter["precision"] - 2 / 7) < 1e-12, "필터 후 고유 관계는 7건입니다."
assert before_filter["precision"] == before_filter["recall"] == before_filter["f1"], "이 단계는 세 값이 모두 2/7로 같습니다."
assert (after_filter["predicted"], after_filter["tp"], after_filter["fp"], after_filter["fn"]) == (8, 2, 6, 5),             "교정으로 한 관계가 필터를 다시 통과합니다."
assert abs(after_filter["precision"] - 0.25) < 1e-12 and abs(after_filter["f1"] - 4 / 15) < 1e-12,             "되살아난 관계가 오답이라 두 값이 내려갑니다."
assert after_filter["recall"] == before_filter["recall"], "되살아난 관계는 정답이 아니라 재현율이 그대로입니다."
print("✅ 교정 전후 재평가 통과!")

## 9. 필터와 교정이 어떤 관계를 바꿨는지 찾으세요

**배경**: 총점만 보면 무엇이 사라지고 무엇이 되살아났는지 알 수 없습니다. 교안 02의 4-2절처럼 집합으로 확인합니다.

**요구사항**

- **`before_filter_keys`**, **`after_filter_keys`**를 각각 `triples`와 `corrected_triples`에서 스키마 통과, 평가 범위, 근거 원문 일치를 모두 거친 행의 triple_key 집합으로 만드세요.  
- **`filter_effect`**를 딕셔너리로 만드세요. **`removed_false_positives`**는 근거 필터로 사라진 오답, **`lost_true_positives`**는 근거 필터로 사라진 정답의 정렬된 리스트입니다. 두 리스트의 원소는 triple_key 튜플 그대로 둡니다. 비교 기준은 필터 전 집합인 predicted_keys입니다.  
- **`fix_effect`**를 딕셔너리로 만드세요. **`new_true_positives`**는 교정으로 새로 들어온 정답, **`new_false_positives`**는 교정으로 되살아난 오답, **`lost_true_positives`**는 교정으로 사라진 정답의 정렬된 리스트입니다. 세 리스트의 원소도 triple_key 튜플 그대로 둡니다.  
- 각 목록의 건수와 관계 이름을 출력하세요.

**확인 기준**: 필터가 없앤 오답은 (MTX, BINDS, FPGS) 1관계이고 사라진 정답은 0관계입니다. 교정은 그 오답 1관계를 되살렸고 새 정답과 정답 손실은 0관계입니다. 근거를 고쳐도 이름 표기가 골드와 달라 그 관계는 여전히 FP입니다.

<details><summary>힌트</summary>

**접근방법**: 필터 전, 필터 후, 교정 후 세 집합을 만들고 차집합의 방향을 구분합니다.

**세부구현**

1. 정답과 오답은 gold_keys와의 교집합, 차집합으로 가릅니다.  
2. 필터의 효과는 필터 전 집합에서 필터 후 집합을 뺍니다.  
3. 교정의 효과는 교정 전 필터 후 집합을 기준으로 비교합니다.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert before_filter_keys == {triple_key(row) for row in scoped
                             if row["evidence"].strip() and row["evidence"] in docs[row["source_doc_id"]]["text"]},             "교정 전 집합은 원본 triples에서 세 검사를 모두 거친 행입니다."
assert after_filter_keys == before_filter_keys | {("MTX", "BINDS", "FPGS")}, "교정본을 넣었는지 확인하세요."
assert len(before_filter_keys) == 7 and len(after_filter_keys) == 8, "교정으로 고유 관계가 하나 늘어납니다."
assert filter_effect["removed_false_positives"] == [("MTX", "BINDS", "FPGS")],             "필터 전 오답에서 필터 후 집합을 빼고 정렬하세요."
assert filter_effect["lost_true_positives"] == [], "이번 필터는 정답을 하나도 지우지 않았습니다."
assert fix_effect["new_false_positives"] == [("MTX", "BINDS", "FPGS")],             "교정 효과는 교정 전 필터 후 집합을 기준으로 빼세요."
assert fix_effect["new_true_positives"] == [] and fix_effect["lost_true_positives"] == [],             "되살아난 관계는 골드에 없으므로 새 정답이 아닙니다."
assert ("Methotrexate", "BINDS", "folylpolyglutamate synthase") in fn_keys, "이름 표기가 달라 정답으로 세지 않습니다."
print("✅ 필터·교정 효과 통과!")

## 10. 골드 변경과 추출기 개선을 구분하세요

**배경**: 평가 도중 골드를 바꾸면 개선 전후가 같은 시험을 치른 것이 아니게 됩니다.

**요구사항**

- **판정 메모**에 다음 상황을 설명하세요. 팀원이 Corticosteroids도 개체로 인정하자고 제안했습니다.  
- 원래 지침을 유지할 때 해당 항목의 상태를 적으세요.  
- 제안을 받아들일 때 바꿔야 하는 주석 지침, 골드, 평가 버전을 적으세요. 관계 수만 더하고 끝내면 안 되는 이유도 설명하세요.  
- 기존 점수와 새 기준 점수를 같은 개선 전후 표에 직접 넣어도 되는지 판단하세요.  
- 점수가 어느 방향으로 얼마나 변하는지는 실제 전체 재계산 전까지 단정하지 마세요.

**확인 기준**: 지침을 바꾸는 의사결정과 추출을 고치는 실험을 구분하며, 기준 변경 시 양쪽 결과를 새 기준으로 다시 평가합니다. 자동 채점은 하지 않습니다. 작성한 뒤 정답 노트북의 모범 서술과 비교하세요.

<details><summary>힌트</summary>

**접근방법**: 고정할 것과 바꿀 것을 나눈 뒤 두 종류의 변경이 섞이지 않게 설명하세요.

**세부구현**

1. 현재 지침에서 그 항목이 어떤 상태인지 먼저 적으세요.  
2. 기준을 바꿀 때 함께 다시 만들어야 하는 산출물을 나열하세요.  
3. 서로 다른 기준으로 잰 점수를 한 표에 넣을 수 있는지 판단하세요.

</details>

*(여기에 원문 근거와 판단을 작성하세요.)*

## 11. 좋아진 점과 나빠진 점을 함께 보고 다음 실험을 정하세요

**배경**: 이번에는 실제로 한 항목을 교정하고 같은 골드로 재평가했습니다. 계산한 값을 근거로 판단합니다.

**요구사항**

- **판단 보고**에 교정으로 좋아진 값과 나빠진 값을 각각 숫자로 적으세요. 근거 원문 일치율과 필터 후 P/R/F1을 사용합니다.  
- 필터 후 정밀도가 왜 떨어졌는지 `fix_effect`의 관계 이름으로 설명하세요.  
- 근거 원문 일치율만 보고 품질이 좋아졌다고 보고하면 안 되는 이유를 적으세요.  
- **다음 실험**을 한 가지만 고르고 `가설 / 바꿀 것 한 가지 / 고정 조건 / 평가 방법 / 채택 기준 / 현재 상태`로 적으세요. 이름 표기를 통일하는 작업은 다음 단원의 내용이므로 이번 실험에 섞지 않습니다.  
- 개선판의 **새 추출 전체**를 같은 범위로 걸러 measure_exact로 평가한다고 적으세요. 옛 표본에 남은 키만 고르지 않습니다.

**확인 기준**: 숫자와 관계 이름을 함께 근거로 쓰고, 한 지표가 올랐다는 사실만으로 채택을 결정하지 않습니다. 자동 채점은 하지 않습니다. 작성한 뒤 정답 노트북의 모범 서술과 비교하세요.

<details><summary>힌트</summary>

**접근방법**: 계산한 숫자를 먼저 적고 그 숫자가 왜 그렇게 움직였는지 관계 이름으로 설명하세요.

**세부구현**

1. 좋아진 값과 나빠진 값을 분모까지 함께 적으세요.  
2. 변화의 원인이 된 관계 이름을 fix_effect에서 찾아 인용하세요.  
3. 다음 실험은 바꾸는 조건이 하나인지 확인하세요.

</details>

*(여기에 원문 근거와 판단을 작성하세요.)*

## 12. 기준선, 교정 결과, 다음 상태를 보고서로 저장하세요

**배경**: 검토자가 무엇을 실제로 측정했고 무엇이 계획인지 알 수 있어야 합니다.

**요구사항**

- **`quality_report`**를 딕셔너리로 만드세요. **`evaluation`**에는 evaluation_plan, **`before`**에는 before_scores, **`after`**에는 after_scores를 넣습니다.  
- **`review`**는 `{"fp_unique": fp_keys의 크기, "fp_evidence_rows": fp_evidence의 행 수, "fn_unique": fn_keys의 크기, "duplicate_rows": duplicate_rows}` 딕셔너리입니다.  
- **`correction`**에는 quote_change를 넣고, **`effects`**에는 `{"filter": filter_effect, "fix": fix_effect}`를 넣으세요. 튜플은 저장하기 전에 리스트로 바꿉니다.  
- **`conflict_check`**는 `{"pairs": conflict_pairs를 리스트의 리스트로 바꾼 값, "probe_candidates": 가상 기록의 후보 수, "scoped_candidates": 실제 추출의 후보 수, "notes": conflict_notes}` 딕셔너리입니다.  
- **`next_experiment`**는 `"미실행"`, **`decision`**에는 채택 여부를 본인 문장으로 스무 자 이상 적으세요.  
- quality_report를 **`lv2_quality_report.json`**, fp_evidence를 **`lv2_fp_evidence.json`**, fn_evidence를 **`lv2_fn_evidence.json`**으로 저장하고, corrected_triples는 제공된 write_rows로 **`lv2_corrected_triples.jsonl`**에 저장하세요.

**확인 기준**: 저장한 before의 필터 후 F1은 2/7, after의 필터 후 F1은 4/15입니다. 교정 기록과 두 효과 목록이 함께 남고 다음 실험은 미실행으로 표시됩니다. decision은 스무 자 이상입니다.

<details><summary>힌트</summary>

**접근방법**: 계산한 값과 교정 기록을 한 파일에 모으되 실행하지 않은 결과는 비워 둡니다.

**세부구현**

1. 지표의 원래 값을 반올림 없이 저장하세요.  
2. 집합과 튜플은 JSON으로 저장하기 전에 리스트로 바꾸세요.  
3. write_json으로 저장한 파일을 다시 읽어 확인하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert quality_report["evaluation"] == evaluation_plan, "1번에서 만든 평가 계획을 그대로 넣으세요."
assert quality_report["before"] == before_scores and quality_report["after"] == after_scores,             "8번의 두 결과를 반올림 없이 그대로 넣으세요."
assert quality_report["review"] == {"fp_unique": len(fp_keys), "fp_evidence_rows": len(fp_evidence),
                                    "fn_unique": len(fn_keys), "duplicate_rows": duplicate_rows},             "고유 관계 수와 근거 행 수를 구분해 넣으세요."
assert quality_report["correction"] == quote_change, "7번의 변경 기록을 그대로 넣으세요."
assert quality_report["effects"]["filter"]["removed_false_positives"] == [["MTX", "BINDS", "FPGS"]],             "저장 전에 튜플을 리스트로 바꾸세요."
assert quality_report["effects"]["fix"]["new_false_positives"] == [["MTX", "BINDS", "FPGS"]],             "fix 아래에도 9번의 교정 효과를 리스트로 바꿔 넣으세요."
assert quality_report["conflict_check"]["scoped_candidates"] == 0, "후보가 0건이어도 그대로 기록합니다."
assert quality_report["conflict_check"]["probe_candidates"] == 1, "가상 기록의 후보 수도 함께 남기세요."
assert quality_report["next_experiment"] == "미실행", "새 프롬프트로 다시 추출하지 않았으므로 미실행입니다."
assert isinstance(quality_report["decision"], str) and len(quality_report["decision"].strip()) >= 20,             "채택 여부를 한 문장 이상으로 적으세요."
saved = json.loads((output_dir / "lv2_quality_report.json").read_text())
assert saved == quality_report, "파일 내용까지 확인하세요."
assert abs(saved["before"]["after_quote_filter"]["f1"] - 2 / 7) < 1e-12, "저장한 값이 계산한 값과 달라졌습니다."
assert abs(saved["after"]["after_quote_filter"]["f1"] - 4 / 15) < 1e-12, "저장한 값이 계산한 값과 달라졌습니다."
assert json.loads((output_dir / "lv2_fp_evidence.json").read_text()) == fp_evidence, "fp_evidence를 파일로 저장하세요."
assert json.loads((output_dir / "lv2_fn_evidence.json").read_text()) == fn_evidence, "fn_evidence를 파일로 저장하세요."
saved_rows = [json.loads(line) for line
              in (output_dir / "lv2_corrected_triples.jsonl").read_text().splitlines() if line.strip()]
assert saved_rows == corrected_triples, "교정본 43행을 그대로 저장하세요."
print("✅ LV2 보고서·교정본 저장 통과!")